# code-server — minimal notebook

Fetch, install, start. The scripts are pulled from the repo instead of being
embedded, so this notebook stays ~100 lines instead of 1,650.

**Runtime → Run all.** No credentials needed: with no ngrok token it falls back
to a Cloudflare quick tunnel.

> Colab's FAQ does not permit remote-desktop / remote-IDE use, and Colab warns
> when it detects one. Read this as a convenience for a host you control, not as
> something to run on Colab indefinitely.

## 1. Fetch the scripts

In [ ]:
import pathlib, stat, urllib.request

REF  = "main"                       # pin a commit SHA to bypass the raw CDN cache
REPO = "recursive-factory/code-server"
BASE = pathlib.Path("/content/cs")

for rel in ["lib/common.sh", "install.sh", "serve.sh", "supervise.sh", "stop.sh", "doctor.sh"]:
    url = f"https://raw.githubusercontent.com/{REPO}/{REF}/colab/{rel}"
    f = BASE / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(urllib.request.urlopen(url, timeout=30).read().decode())
    if f.suffix == ".sh":
        f.chmod(f.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

print("fetched into", BASE, "-", REF)

## 2. Configure (optional)

Everything has a working default — edit only what you need.

| Variable | Default | Notes |
|---|---|---|
| `TUNNEL` | `auto` | ngrok when a token exists, else a Cloudflare quick tunnel (no account). Set `"tailscale"` for a **fixed** `*.ts.net` URL (free plan; needs `TS_AUTHKEY`) |
| `CS_PORT` | auto | pin a port, or leave empty to scan `CS_PORT_RANGE` and take the first free one. The tunnel always maps to whatever is chosen |
| `CS_PORT_RANGE` | `3000-5000` | scan range used when `CS_PORT` is empty. Avoids 8080, which `datalab/web/app.js` owns on some Colab runtimes |
| `NGROK_DOMAIN` | — | reserved ngrok domain, so the URL survives restarts |
| `IDLE` | `1800` | shut down after this many seconds with no connections |
| `MAX_RUNTIME` | `39600` | hard stop at 11 h |

The resolved port is recorded in `$CS_STATE_DIR/port`, so `stop.sh`, the idle
watchdog and `doctor.sh` always operate on the port actually in use.


In [ ]:
import os

# Colab Secrets are NOT environment variables. The only way in is:
#     from google.colab import userdata
#     userdata.get('TS_AUTHKEY')
# Precedence here: this cell -> real environment variable -> Colab Secret.
NGROK_TOKEN  = ""
TS_AUTHKEY   = ""     # tskey-auth-... (reusable, see README)

TUNNEL       = ""     # "" = auto | "ngrok" | "cloudflared" | "tailscale" | "none"
TS_HOSTNAME  = ""     # "" = "code" -> https://code.<tailnet>.ts.net; keep it constant
NGROK_DOMAIN = ""     # reserved domain -> stable URL, e.g. "me.ngrok-free.app"
CS_PORT      = ""     # "" = auto: first free port in CS_PORT_RANGE, else pin one e.g. 3225
CS_PORT_RANGE = ""    # "" = 3000-5000
IDLE         = 1800   # stop after 30 min with no connections
MAX_RUNTIME  = 39600  # hard stop at 11 h

def _secret(name):
    """Read a Colab Secret. Never raises; explains the common misconfigurations."""
    try:
        from google.colab import userdata
    except Exception:
        return ""                      # not running in Colab
    try:
        return (userdata.get(name) or "").strip()
    except Exception as e:
        msg = f"{type(e).__name__}: {e}".lower()
        if "notebook" in msg and "access" in msg:
            print(f"[secret] '{name}' exists but notebook access is off — "
                  f"open the Secrets panel (key icon) and grant this notebook access")
        elif "notfound" in msg or "not found" in msg:
            pass                        # simply not configured; stay quiet
        else:
            print(f"[secret] '{name}': {type(e).__name__}: {str(e)[:110]}")
        return ""

def _resolve(cell_value, name):
    return cell_value.strip() or os.environ.get(name, "").strip() or _secret(name)

tok   = _resolve(NGROK_TOKEN, "NGROK_TOKEN")
tskey = _resolve(TS_AUTHKEY, "TS_AUTHKEY")

# Only write back when non-empty: an unconditional assignment would clobber a
# key that was exported as a real environment variable.
if tok:
    os.environ["NGROK_TOKEN"] = tok
if tskey:
    os.environ["TS_AUTHKEY"] = tskey

os.environ["CS_TUNNEL"] = TUNNEL
os.environ["CS_IDLE_TIMEOUT"] = str(IDLE)
os.environ["CS_MAX_RUNTIME"] = str(MAX_RUNTIME)
os.environ["CS_PORT"] = str(CS_PORT)
if CS_PORT_RANGE:
    os.environ["CS_PORT_RANGE"] = CS_PORT_RANGE
if NGROK_DOMAIN:
    os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN
if TS_HOSTNAME:
    os.environ["TS_HOSTNAME"] = TS_HOSTNAME

def mask(s):
    return f"{s[:13]}…({len(s)} chars)" if s else "none"

print("tunnel:", TUNNEL or "auto", "| port:", CS_PORT or f"auto ({CS_PORT_RANGE or '3000-5000'})")
print("ngrok token:", mask(tok), "| tailscale key:", mask(tskey))


## 3. Install (1–3 min on a fresh runtime)

In [ ]:
import os, subprocess

# Install tailscale when it will be used: TUNNEL says so, or a key is
# present and the auto order would pick it.
want_ts = "1" if (TUNNEL == "tailscale" or TS_AUTHKEY or os.environ.get("TS_AUTHKEY")) else "0"
extras = {"WITH_CLOUDFLARED": "1", "WITH_TAILSCALE": want_ts}
subprocess.run(["bash", str(BASE / "install.sh")], check=True,
               env=dict(os.environ, **extras))


## 4. Start (returns immediately)

In [ ]:
import os, pathlib, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
STATE.mkdir(parents=True, exist_ok=True)

# stop.sh kills any previous supervisor and reclaims the port, so this is re-runnable
subprocess.run(["bash", str(BASE / "stop.sh")], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
(STATE / "credentials.txt").unlink(missing_ok=True)

with open(STATE / "supervise.out", "w") as fh:
    proc = subprocess.Popen(["bash", str(BASE / "supervise.sh")], stdout=fh,
                            stderr=subprocess.STDOUT, start_new_session=True)
(STATE / "supervise.pid").write_text(str(proc.pid))
print("supervisor pid", proc.pid, "-> log:", STATE / "supervise.out")

## 5. Wait for the URL (live progress)

In [ ]:
import os, time, pathlib
from IPython.display import display, clear_output, HTML

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
creds, log = STATE / "credentials.txt", STATE / "supervise.out"

for t in range(30):
    raw = creds.read_text().strip() if creds.exists() else ""
    info = dict(l.split("=", 1) for l in raw.splitlines() if "=" in l) if "url=" in raw else {}
    clear_output(wait=True)

    url = info.get("url")
    if url:
        pw = info.get("password")
        display(HTML(f'<p style="font-size:17px"><b>code-server</b> '
                     f'<a href="{url}" target="_blank">{url}</a></p>'
                     f'<p><b>password</b> <code>{pw}</code></p>'))
        print("tunnel:", info.get("tunnel"), "| state:", STATE)
        break

    tail = "\n".join(log.read_text().splitlines()[-8:]) if log.exists() else "(starting)"
    print(f"{t}s\n" + "-" * 56 + "\n" + tail)
    time.sleep(1)
else:
    print("no URL yet after 30s. The stack keeps running in the background;\n"
          "re-run this cell to check again, or run the status cell.\n"
          "Log:", log)

## 6. Stop (opt-in)

This cell does nothing unless `STOP = True`, because **Runtime → Run all**
executes it too — which is what killed a previous session one second after its
URL appeared. Set the flag and run only this cell when you want to stop.


In [ ]:
STOP = False        # set True, then run THIS cell alone

import os, pathlib, signal, subprocess

if not STOP:
    print("skipped: 'Run all' executes this cell too, so stopping is opt-in.")
    print("To shut the stack down: set STOP = True and run this cell alone.")
    print("It also stops by itself after CS_IDLE_TIMEOUT with no connections.")
else:
    STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
    pid = STATE / "supervise.pid"
    if pid.exists():
        try:
            os.killpg(os.getpgid(int(pid.read_text().strip())), signal.SIGTERM)
            print("stopped supervisor")
        except (ValueError, ProcessLookupError):
            pass
        pid.unlink(missing_ok=True)
    subprocess.run(["bash", str(BASE / "stop.sh")], check=False)


Diagnostics: `!bash {BASE}/doctor.sh` · logs: `/content/.code-server/`

The scripts come from this repo, so a fix lands here the moment it is
pushed — re-run cell 1 to refresh them.
